# 02 - Datetime Cleaning & Minute to Hour/Day

After the first quality check, I cleaned the date-time column and created useful time-based tables.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
file = "../01_Data/raw/household_power_consumption.csv"
df = pd.read_csv(file, na_values="?")

In [3]:
df["datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    dayfirst=True, errors="coerce"
)

In [4]:
df = df.sort_values("datetime")
df[["Date", "Time", "datetime"]].head()

,Date,Time,datetime
0,16/12/2006,17:24:00,2006-12-16 17:24:00
1,16/12/2006,17:25:00,2006-12-16 17:25:00
2,16/12/2006,17:26:00,2006-12-16 17:26:00
3,16/12/2006,17:27:00,2006-12-16 17:27:00
4,16/12/2006,17:28:00,2006-12-16 17:28:00


## Check the time gaps

In [5]:
gap = df["datetime"].diff().dt.total_seconds().div(60)

In [6]:
print("1-minute rows (%):", round(gap.dropna().eq(1).mean()*100, 2))
print("Largest gap (minutes):", int(gap.max()))

1-minute rows (%): 100.0
Largest gap (minutes): 1


## Numeric columns

In [7]:
cols = ["Global_active_power", "Global_reactive_power", "Voltage",
        "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]

In [8]:
for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

## Missing values for aggregation

I keep missing readings as missing instead of making up values. Coverage is checked after aggregation.

In [ ]:
print("Missing numeric values:", df[cols].isna().sum().sum())


## Create daily data

In [11]:
daily = df.resample("D").agg(
    daily_kwh=("Global_active_power", lambda s: s.sum()/60),
    avg_voltage=("Voltage", "mean"),
    avg_intensity=("Global_intensity", "mean"),
    avg_reactive_power=("Global_reactive_power", "mean")
)

In [12]:
daily["records"] = df["Global_active_power"].resample("D").count()
daily["coverage_pct"] = daily["records"] / 1440 * 100

## Keep good-coverage days

In [13]:
daily = daily[daily["coverage_pct"] >= 90].copy()

In [14]:
daily["dayofweek"] = daily.index.dayofweek
daily["month"] = daily.index.month
daily["is_weekend"] = (daily["dayofweek"] >= 5).astype(int)

## Create hourly data

In [15]:
hourly = df.resample("h").agg(
    energy_kwh=("Global_active_power", lambda s: s.sum()/60),
    avg_voltage=("Voltage", "mean"),
    avg_intensity=("Global_intensity", "mean")
)

In [16]:
hourly["records"] = df["Global_active_power"].resample("h").count()
hourly["coverage_pct"] = hourly["records"] / 60 * 100

In [17]:
hourly = hourly[hourly["coverage_pct"] >= 90].copy()

In [18]:
hourly["hour"] = hourly.index.hour
hourly["dayofweek"] = hourly.index.dayofweek
hourly["month"] = hourly.index.month
hourly["is_weekend"] = (hourly["dayofweek"] >= 5).astype(int)

## Save the two tables

In [19]:
daily.to_csv("../01_Data/processed/daily_energy.csv", index_label="date")
hourly.to_csv("../01_Data/processed/hourly_energy.csv", index_label="date_time")

In [20]:
print("Daily rows:", len(daily))
print("Hourly rows:", len(hourly))

Daily rows: 725
Hourly rows: 17410


Now the raw minute data is ready for analysis.